# DX 704 Week 7 Project

This week's project will investigate issues in a quadcopter controller based using a linear quadratic regulator.
You will start with an existing model of the system and logs from a quadcopter based on it, investigate discrepancies, and ultimately train a new model of the system dynamics.

The full project description and a template notebook are available on GitHub: [Project 7 Materials](https://github.com/bu-cds-dx704/dx704-project-07).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Introduction

You've just joined a drone startup.
On your first day, you join your new team to watch a test flight for a new quadcopter prototype.
Watching the prototype fly, the team comments that it is not as smooth as usual and suspects that something is off in the controller.
They provide you a copy of the dynamics model and log data from the prototype to investigate.

The quadcopter control model is a slightly more sophisticated version of the 1D quadcopter problem previously considered.

The state vector $\mathbf{x}_t$ now includes an acceleration component, and the action vector now has a servo control for the throttle instead of a raw force component.
\begin{array}{rcl}
\mathbf{x}_t & = & \begin{bmatrix} x_t \\ v_t \\ a_t \end{bmatrix} \\
\mathbf{u_t} & = & \begin{bmatrix} u_t \end{bmatrix}
\end{array}

## Part 1: Reconstruct the Control Matrix

You are provided the dynamics model in the files `model-A.tsv`, `model-B.tsv`, `cost-Q.tsv` and `cost-R.tsv`.
Recompute the control matrix $\mathbf{K}$ to be used in the infinite horizon linear quadratic regulator problem.

In [1]:
import numpy as np
import pandas as pd
from scipy.linalg import solve_discrete_are

# Load model matrices
A = pd.read_csv("model-A.tsv", sep="\t", index_col=0).values.astype(float)
B = pd.read_csv("model-B.tsv", sep="\t", index_col=0).values.astype(float)
Q = pd.read_csv("cost-Q.tsv", sep="\t", index_col=0).values.astype(float)
R = pd.read_csv("cost-R.tsv", sep="\t", index_col=0).values.astype(float)

print("A =\n", A)
print("B =\n", B)
print("Q =\n", Q)
print("R =\n", R)

# Solve the Discrete Algebraic Riccati Equation (DARE)
P = solve_discrete_are(A, B, Q, R)

# Compute the LQR control matrix K
# K = (R + B' P B)^{-1} B' P A
K = np.linalg.inv(R + B.T @ P @ B) @ (B.T @ P @ A)

print("\nK =\n", K)


A =
 [[1. 1. 0.]
 [0. 1. 1.]
 [0. 0. 1.]]
B =
 [[0.]
 [0.]
 [1.]]
Q =
 [[5. 0. 0.]
 [0. 1. 0.]
 [0. 0. 2.]]
R =
 [[5.]]

K =
 [[0.33445985 1.30445413 1.85813088]]


Save $\mathbf{K}$ in a file "control-K-intended.tsv" with columns x, v and a.

In [2]:
# Save K to file with columns x, v, a
K_df = pd.DataFrame(K, columns=["x", "v", "a"], index=["u"])
K_df.index.name = "index"
K_df.to_csv("control-K-intended.tsv", sep="\t")
print("Saved control-K-intended.tsv")
print(K_df)


Saved control-K-intended.tsv
             x         v         a
index                             
u      0.33446  1.304454  1.858131


Submit "control-K-intended.tsv" in Gradescope.

## Part 2: Recompute the Actions for the Logged States

You get access to the log data for the prototype as the file "qc-log.tsv".
It conveniently saves all the state and actions made.
Recompute the actions based on your $\mathbf{K}$ matrix computed in part 1.

In [3]:
# Load log data
log = pd.read_csv("qc-log.tsv", sep="\t", index_col=0)
print(log.head())

# Recompute actions using K: u = -K @ x
X = log[["x", "v", "a"]].values
u_check = -(K @ X.T).flatten()

print("\nFirst 5 logged actions:", log["u"].values[:5])
print("First 5 recomputed actions:", u_check[:5])


              x         v         a         u
index                                        
0     -5.000000  0.000000  0.000000  1.702188
1     -5.000000 -0.017022  1.531969 -1.263200
2     -5.018724  1.452683  0.548285 -1.249321
3     -3.420773  1.840779 -0.521275 -0.212127
4     -1.395916  1.163611 -0.764317  0.452895

First 5 logged actions: [ 1.70218775 -1.26320045 -1.24932102 -0.21212738  0.45289543]
First 5 recomputed actions: [ 1.67229926 -1.15209535 -1.23518258 -0.2885035   0.36920157]


Save your computed actions as "qc-check.tsv" with columns "index" and "u_check".

In [4]:
# Save computed actions with columns "index" and "u_check"
qc_check = pd.DataFrame({"index": log.index, "u_check": u_check})
qc_check.to_csv("qc-check.tsv", sep="\t", index=False)
print("Saved qc-check.tsv")
print(qc_check.head())


Saved qc-check.tsv
   index   u_check
0      0  1.672299
1      1 -1.152095
2      2 -1.235183
3      3 -0.288504
4      4  0.369202


Submit "qc-check.tsv" in Gradescope.

## Part 3: Reverse Engineer the Actual Control Matrix

Now that you have found a systematic difference between your computed actions and the logged actions, estimate $
$, the control matrix that was actually used to choose actions in the prototype.

Hint: With a linear quadratic regulator, the optimal actions are always linear combinations of the state that are calculated using the control matrix.
You can use linear regression to reverse-engineer the coefficients in the control matrix.

In [5]:
from sklearn.linear_model import LinearRegression

# Use linear regression to estimate K_actual from logged data
# u_t = -K_actual @ x_t  =>  u_t = [x_t, v_t, a_t] @ (-K_actual.T)
X_log = log[["x", "v", "a"]].values
u_log = log["u"].values

reg = LinearRegression(fit_intercept=False)
reg.fit(X_log, u_log)

# The fitted coefficients are -K_actual (one row since u is scalar)
K_actual = -reg.coef_.reshape(1, -1)
print("K_actual =\n", K_actual)
print("\nK_intended =\n", K)
print("\nDifference (K_actual - K_intended):\n", K_actual - K)


K_actual =
 [[0.34043755 1.30012023 1.95011696]]

K_intended =
 [[0.33445985 1.30445413 1.85813088]]

Difference (K_actual - K_intended):
 [[ 0.0059777  -0.0043339   0.09198608]]


Save $\mathbf{K}_{\mathrm{actual}}$ in "control-K-actual.tsv" with the same format as "control-K-intended.tsv".

In [6]:
# Save K_actual with same format as control-K-intended.tsv
K_actual_df = pd.DataFrame(K_actual, columns=["x", "v", "a"], index=["u"])
K_actual_df.index.name = "index"
K_actual_df.to_csv("control-K-actual.tsv", sep="\t")
print("Saved control-K-actual.tsv")
print(K_actual_df)


Saved control-K-actual.tsv
              x        v         a
index                             
u      0.340438  1.30012  1.950117


Submit "control-k-actual.tsv" in Gradescope.

## Part 4: Recompute the System Dynamics from the Log Data

On further investigation, it turns out that the control matrix $\mathbf{K}$ in the prototype was modified to compensate for state prediction errors.
You would like to recompute the $\mathbf{A}$ and $\mathbf{B}$ matrices using the log data since they are used to predict the next states.
However, since the action vector $\mathbf{u}_t$ is linearly dependent on the state via $\mathbf{u}_t=-\mathbf{K} \mathbf{x}_t$, you need a new data set so you can separate the effects of the $\mathbf{A}$ and $\mathbf{B}$ matrices.
Your co-workers kindly provide a new file "qc-train.tsv" where noise was added to each action.
Estimate the true $\mathbf{A}$ and $\mathbf{B}$ matrices based on this file.

In [7]:
# Load training data with noisy actions
train = pd.read_csv("qc-train.tsv", sep="\t", index_col=0)
print(train.head())

# Build (x_t, u_t) -> x_{t+1} pairs
# x_{t+1} = A x_t + B u_t
# Stack features as [x_t, u_t] (shape: n_samples x 4) and predict x_{t+1} (shape: n_samples x 3)

states = train[["x", "v", "a"]].values
actions = train["u"].values

# Consecutive pairs: features from row t, target from row t+1
X_train = np.hstack([states[:-1], actions[:-1].reshape(-1, 1)])   # [x_t, v_t, a_t, u_t]
Y_train = states[1:]                                                 # [x_{t+1}, v_{t+1}, a_{t+1}]

# Fit multioutput linear regression (no intercept: x_{t+1} = A x_t + B u_t)
reg_dyn = LinearRegression(fit_intercept=False)
reg_dyn.fit(X_train, Y_train)

# Coefficients shape: (3 outputs, 4 features) -> split into A (3x3) and B (3x1)
coef = reg_dyn.coef_          # shape (3, 4)
A_new = coef[:, :3]
B_new = coef[:, 3:]

print("\nA_new =\n", A_new)
print("\nA_original =\n", A)
print("\nB_new =\n", B_new)
print("\nB_original =\n", B)


              x         v         a         u
index                                        
0     -5.000000  0.000000  0.000000  1.729856
1     -5.000000 -0.017299  1.556871 -1.311911
2     -5.019028  1.476577  0.531837 -1.198850
3     -3.394793  1.846154 -0.493944 -0.297565
4     -1.364024  1.195267 -0.811147  0.472619

A_new =
 [[ 1.00000000e+00  1.10000000e+00  2.88657986e-15]
 [-1.28195498e-16  9.00000000e-01  9.50000000e-01]
 [-9.31378031e-17  5.55111512e-16  1.10000000e+00]]

A_original =
 [[1. 1. 0.]
 [0. 1. 1.]
 [0. 0. 1.]]

B_new =
 [[-1.11022302e-16]
 [-1.00000000e-02]
 [ 9.00000000e-01]]

B_original =
 [[0.]
 [0.]
 [1.]]


Save $\mathbf{A}$ and $\mathbf{B}$ to "model-A-new.tsv" and "model-B-new.tsv" respectively.

In [8]:
# Save A_new with row/column labels x, v, a
A_new_df = pd.DataFrame(A_new, columns=["x", "v", "a"], index=["x", "v", "a"])
A_new_df.index.name = "index"
A_new_df.to_csv("model-A-new.tsv", sep="\t")
print("Saved model-A-new.tsv")
print(A_new_df)

# Save B_new with column label u and row labels x, v, a
B_new_df = pd.DataFrame(B_new, columns=["u"], index=["x", "v", "a"])
B_new_df.index.name = "index"
B_new_df.to_csv("model-B-new.tsv", sep="\t")
print("\nSaved model-B-new.tsv")
print(B_new_df)


Saved model-A-new.tsv
                  x             v             a
index                                          
x      1.000000e+00  1.100000e+00  2.886580e-15
v     -1.281955e-16  9.000000e-01  9.500000e-01
a     -9.313780e-17  5.551115e-16  1.100000e+00

Saved model-B-new.tsv
                  u
index              
x     -1.110223e-16
v     -1.000000e-02
a      9.000000e-01


Submit "model-A-new.tsv" and "model-B-new.tsv" in Gradescope.

## Part 5: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 6: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.